# Download Code Data (The Stack v2)
Downloads code from BigCode's The Stack v2 on HuggingFace.
Saves as .txt files into `data_code/` with `<|endoftext|>` separators.
Resumable — detects existing files and skips ahead.

In [ ]:
!pip install -q datasets huggingface_hub
from huggingface_hub import login

# Paste your HuggingFace token here (needs access to gated datasets)
# Get one at: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <-- paste your token

if not HF_TOKEN:
    HF_TOKEN = input("Enter your HuggingFace token: ").strip()

login(token=HF_TOKEN)
print("Logged in to HuggingFace")

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/sparkyllm/datasets_pretrain/data_code"
os.makedirs(OUT_DIR, exist_ok=True)

# ================= CONFIGURATION =================
# Using The Stack v1 (dedup) — has actual code content
# Languages use lowercase with data_dir="data/{lang}"
LANGUAGES = [
    "python",
    "javascript",
    "java",
    "c",
    "c++",
    "go",
    "rust",
    "typescript",
    "shell",
    "sql",
]

DOCS_PER_LANGUAGE = 50_000     # files per language
DOCS_PER_FILE = 5_000          # code files per output .txt
MIN_LENGTH = 200               # skip very short files
MAX_LENGTH = 50_000            # skip very long files
SEPARATOR = "\n<|endoftext|>\n"

print(f"Dataset: bigcode/the-stack-dedup")
print(f"Languages: {', '.join(LANGUAGES)}")
print(f"{DOCS_PER_LANGUAGE:,} files per language, {len(LANGUAGES) * DOCS_PER_LANGUAGE:,} total")
print(f"Output: {OUT_DIR}")

In [ ]:
# ================= DOWNLOAD =================
total_files_written = 0
total_docs = 0
total_chars = 0

for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Language: {lang}")
    print(f"{'='*60}")

    # Check what we already have for this language
    lang_tag = lang.replace("+", "p")  # c++ -> cpp for filenames
    existing = sorted([
        f for f in os.listdir(OUT_DIR)
        if f.startswith(f"code_{lang_tag}_") and f.endswith(".txt")
    ])
    if existing:
        last_num = int(existing[-1].replace(f"code_{lang_tag}_", "").replace(".txt", ""))
        docs_done = (last_num + 1) * DOCS_PER_FILE
        if docs_done >= DOCS_PER_LANGUAGE:
            print(f"  Already done ({len(existing)} files). Skipping.")
            continue
        print(f"  Resuming from {docs_done:,} docs ({len(existing)} files)...")
    else:
        docs_done = 0

    try:
        ds = load_dataset(
            "bigcode/the-stack-dedup",
            split="train",
            data_dir=f"data/{lang}",
            streaming=True,
        )
    except Exception as e:
        print(f"  Failed to load {lang}: {e}")
        continue

    buf = []
    file_idx = len(existing)
    lang_docs = 0
    lang_chars = 0
    skipped = 0

    for i, example in enumerate(ds):
        # Skip already-done docs
        if i < docs_done:
            if i % 50_000 == 0 and i > 0:
                print(f"  Skipping... {i:,}/{docs_done:,}")
            continue

        if lang_docs >= DOCS_PER_LANGUAGE:
            break

        content = example.get("content", "").strip()

        # Filter by length
        if len(content) < MIN_LENGTH or len(content) > MAX_LENGTH:
            skipped += 1
            continue

        buf.append(content)
        lang_docs += 1

        if len(buf) >= DOCS_PER_FILE:
            path = os.path.join(OUT_DIR, f"code_{lang_tag}_{file_idx:03d}.txt")
            file_content = SEPARATOR.join(buf)
            with open(path, "w", encoding="utf-8") as f:
                f.write(file_content)
            size_mb = os.path.getsize(path) / 1024 / 1024
            lang_chars += len(file_content)
            print(f"  {path.split('/')[-1]}: {size_mb:.1f} MB ({lang_docs:,} docs, {skipped:,} skipped)")
            buf = []
            file_idx += 1

    # Write remainder
    if buf:
        path = os.path.join(OUT_DIR, f"code_{lang_tag}_{file_idx:03d}.txt")
        file_content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(file_content)
        lang_chars += len(file_content)
        file_idx += 1

    lang_mb = lang_chars / 1024 / 1024
    total_chars += lang_chars
    total_docs += lang_docs
    total_files_written += file_idx - len(existing)
    print(f"  {lang}: {lang_docs:,} docs, {lang_mb:.0f} MB, {skipped:,} skipped")

total_gb = total_chars / 1024 / 1024 / 1024
print(f"\n{'='*60}")
print(f"Done!")
print(f"  {total_docs:,} code files across {len(LANGUAGES)} languages")
print(f"  {total_files_written} new .txt files, ~{total_gb:.1f} GB")
print(f"  Saved to: {OUT_DIR}")
print(f"\nNext: run tokenizer_pipeline -> pre_train")